In [19]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [20]:
df = pd.read_csv("../data/processed/messages_with_clusters.csv")

df = df[df["comment_clean"].notna()]
df.head()


,id,date,date_unixtime,professor_id,professor_name_clean,department,course_name_clean,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,grading_status,attendance_status,comment_clean,sentiment,cluster
0,26,2021-09-05T02:19:28,1630792168,13,داریم n nلطفا استاد هایی که میخواید معرفی کنید...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,نامشخص,نامشخص,سنجی u200cها بیان کننده u200cی توان علمی اساتی...,neutral,2
1,55,2021-09-05T15:17:17,1630838837,32,کلاس داشته n بهمن 98 n nتوضیحات n در مجموع سخت...,NaN,5 n نحوه مدیریت کلاس نظم و زمان 8 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,سختگیر,نامشخص,خودش n nبرای ثبت معرفی استاد به ربات زیر پیام ...,neutral,0
2,66,2021-09-05T15:43:56,1630840436,40,ی خوب گفتن و همه مخالفن چیو نشون میده nیا برعک...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,آسان,نامشخص,مخالف داره یعنی چی n nمثلا معرفی ریپلای شده رو...,neutral,2
3,76,2021-09-05T16:12:31,1630842151,48,کلاس داشته n بهمن 99 n nتوضیحات n خودش تدریس ن...,NaN,8 n نحوه مدیریت کلاس نظم و زمان 3 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,نامشخص,نامشخص,ی n type hashtag text مهندسی_عمران n زمین شناس...,neutral,1
4,85,2021-09-05T21:17:39,1630860459,57,کلاس داشته n بهمن 99 n nتوضیحات n فقط کافیه تو...,NaN,9 n نحوه مدیریت کلاس نظم و زمان 9 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,منصفانه,نامشخص,سنجی یا آزمون میذارن و چند ثانیه فرصت داری جوا...,neutral,2


In [21]:
prof_text = (
    df.groupby("professor_id")["comment_clean"]
    .apply(lambda x: " ".join(x.dropna()))
    .reset_index()
)
prof_meta = (
    df.groupby("professor_id")
    .agg({
        "professor_name_clean": "first",
        "course_name_clean": lambda x: ", ".join(x.dropna().unique()[:3]),
        "sentiment": lambda x: (x == "positive").mean(),
        "grading_status": lambda x: x.value_counts().index[0]
    })
    .reset_index()
)
prof_profiles = prof_text.merge(prof_meta, on="professor_id")
prof_profiles.head()


,professor_id,comment_clean,professor_name_clean,course_name_clean,sentiment,grading_status
0,0,ات درباره اساتید است,None,,0.0,نامشخص
1,5,یه کدگذاری_تاریخ ریاضیات n nمنابع آموزش n فایل...,کلاس داشته n بهمن 99 n nتوضیحات n چیزی اضافه ا...,10 n نحوه مدیریت کلاس نظم و زمان 10 n پاسخگویی...,0.0,منصفانه
2,13,سنجی u200cها بیان کننده u200cی توان علمی اساتی...,داریم n nلطفا استاد هایی که میخواید معرفی کنید...,,0.0,نامشخص
3,32,خودش n nبرای ثبت معرفی استاد به ربات زیر پیام ...,کلاس داشته n بهمن 98 n nتوضیحات n در مجموع سخت...,5 n نحوه مدیریت کلاس نظم و زمان 8 n پاسخگویی ح...,0.0,سختگیر
4,40,مخالف داره یعنی چی n nمثلا معرفی ریپلای شده رو...,ی خوب گفتن و همه مخالفن چیو نشون میده nیا برعک...,,0.0,آسان


In [22]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(prof_profiles["comment_clean"])
cos_sim = cosine_similarity(X, X)


In [23]:
def recommend_similar_professors(professor_id, top_n=5):
    idx = prof_profiles.index[
        prof_profiles["professor_id"] == professor_id
    ][0]

    sim_scores = list(enumerate(cos_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1: top_n + 1]
    indices = [i[0] for i in sim_scores]

    recs = prof_profiles.iloc[indices].copy()
    recs["similarity_score"] = [i[1] for i in sim_scores]

    return recs[
        [
            "professor_name",
            "course_name",
            "rating_avg",
            "sentiment",
            "grading_status_std",
            "similarity_score"
        ]
    ]


In [24]:
def rule_based_recommender(
    min_rating=7.5,
    prefer_easy_grading=True,
    min_sentiment=0.6,
    top_n=10
):
    df_rule = prof_profiles.copy()

    df_rule = df_rule[df_rule["rating_avg"] >= min_rating]
    df_rule = df_rule[df_rule["sentiment"] >= min_sentiment]

    if prefer_easy_grading:
        df_rule = df_rule[
            df_rule["grading_status_std"].isin(["آسان", "منصفانه"])
        ]

    df_rule = df_rule.sort_values(
        by=["rating_avg", "sentiment"],
        ascending=False
    )

    return df_rule.head(top_n)[
        [
            "professor_name",
            "course_name",
            "rating_avg",
            "sentiment",
            "grading_status_std"
        ]
    ]


In [25]:
def hybrid_recommender(
    professor_id=None,
    min_rating=7,
    min_sentiment=0.5,
    top_n=5
):
    base = prof_profiles.copy()

    if professor_id is not None:
        idx = prof_profiles.index[
            prof_profiles["professor_id"] == professor_id
        ][0]
        base["similarity"] = cos_sim[idx]
    else:
        base["similarity"] = 0.5

    base["final_score"] = (
        0.4 * base["rating_avg"] +
        0.4 * base["sentiment"] * 10 +
        0.2 * base["similarity"] * 10
    )

    base = base[
        (base["rating_avg"] >= min_rating) &
        (base["sentiment"] >= min_sentiment)
    ]

    base = base.sort_values("final_score", ascending=False)

    return base.head(top_n)[
        [
            "professor_name",
            "course_name",
            "rating_avg",
            "sentiment",
            "grading_status_std",
            "final_score"
        ]
    ]


In [26]:
prof_profiles["professor_id"].unique()


array([   0,    5,   13,   32,   40,   48,   57,   58,   80,   88,   99,
        102,  110,  118,  125,  130,  133,  134,  138,  140,  145,  178,
        193,  198,  199,  201,  203,  205,  212,  236,  238,  245,  248,
        249,  257,  270,  272,  279,  280,  286,  310,  336,  352,  355,
        368,  376,  397,  398,  401,  411,  414,  416,  418,  420,  430,
        431,  432,  442,  455,  456,  466,  473,  508,  509,  510,  517,
        519,  527,  529,  533,  549,  556,  561,  577,  587,  592,  594,
        595,  603,  644,  656,  663,  678,  692,  695,  696,  707,  709,
        729,  736,  738,  757,  764,  765,  768,  786,  797,  822,  858,
        864,  883,  890,  892,  893,  903,  909,  922,  929,  938,  941,
        953,  958,  968,  971,  976,  981,  988, 1001, 1032, 1035, 1039,
       1046, 1047, 1048, 1050, 1061, 1062, 1074, 1091, 1094, 1099, 1101,
       1105, 1115, 1123, 1128, 1130, 1140, 1154, 1156, 1158, 1163, 1174,
       1180, 1189, 1204, 1208, 1213, 1224, 1229, 12

In [27]:
def recommend_similar_professors(professor_id, top_n=5):
    matches = prof_profiles.index[
        prof_profiles["professor_id"] == professor_id
    ]

    if len(matches) == 0:
        print(f"❌ professor_id={professor_id} not found in profiles")
        return None

    idx = matches[0]

    sim_scores = list(enumerate(cos_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1 : top_n + 1]
    prof_indices = [i[0] for i in sim_scores]

    return prof_profiles.iloc[prof_indices][
        ["professor_id", "professor_name_clean"]
    ]

sample_id = prof_profiles["professor_id"].iloc[0]
recommend_similar_professors(sample_id)



,professor_id,professor_name_clean
419,3428,کلاس داشته n مهر ۱۴۰۳ n nتوضیحات n امتحان های ...
494,3959,کلاس داشته n مهر 1401 n nتوضیحات n سلام nبه نظ...
358,2970,کلاس داشته n بهمن ۴۰۲ n nتوضیحات n استاد استوا...
326,2690,شهین واعظی n type hashtag text معارف n زبان عم...
359,2973,به واسطه نهاد و حضور کلاسی می کنند n nترمی که ...


In [28]:
prof_profiles.to_csv("../data/processed/professor_profiles.csv", index=False)
print("✅ professor_profiles.csv saved")


✅ professor_profiles.csv saved
